# OpenAlice Architecture Gap v1

Direction: `production` / `harness` architecture diagnostic.  
Role: `diagnose`; no production code or model promotion from this notebook.  
Date: `2026-06-02`.  
External reference: [TraderAlice/OpenAlice](https://github.com/TraderAlice/OpenAlice), inspected at commit `a724057e4a39c5a5f227ccf7230803ee9ac5f331`.

## Purpose

The current SOXL repo already has a strong research/validation harness and a growing production scheduler. The user concern is that the project architecture still feels too simple for an end-to-end launchable trading product. This notebook compares the SOXL harness with OpenAlice and records what can be learned without blindly copying a broader multi-asset agent platform.

## Scope

- Compare architecture/control-plane concepts, not strategy performance.
- Keep SOXL single-symbol scope intact.
- Prefer production safety, auditability, and operator workflow over broad multi-asset features.
- Keep conclusions in this notebook; do not update `docs/active_workpad.md` unless a later implementation plan is approved.


In [1]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path
from textwrap import shorten

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "AGENTS.md").exists():
    ROOT = Path("/Users/rziiiii/.openclaw/workspace")
CARD = ROOT / "runtime/production/card/openalice_launch_gap_v1"
CARD.mkdir(parents=True, exist_ok=True)
OPENALICE = Path("/tmp/openalice-ref")
OPENALICE_URL = "https://github.com/TraderAlice/OpenAlice.git"

if not OPENALICE.exists():
    subprocess.run(["git", "clone", "--depth=1", OPENALICE_URL, str(OPENALICE)], check=True)

openalice_head = subprocess.run(
    ["git", "-C", str(OPENALICE), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
soxl_head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT, check=True, capture_output=True, text=True).stdout.strip()
soxl_status = subprocess.run(["git", "status", "--short"], cwd=ROOT, check=True, capture_output=True, text=True).stdout.splitlines()

print(f"soxl_repo={ROOT}")
print(f"soxl_head={soxl_head}")
print(f"dirty_paths={len(soxl_status)}")
print(f"openalice_ref={OPENALICE}")
print(f"openalice_head={openalice_head}")


soxl_repo=/Users/rziiiii/.openclaw/workspace
soxl_head=5eedef532f66f4967578774d6e146b6d861efa00
dirty_paths=19
openalice_ref=/tmp/openalice-ref
openalice_head=a724057e4a39c5a5f227ccf7230803ee9ac5f331


In [2]:
from pathlib import Path

interesting_roots = {
    "soxl_production_entrypoints": ROOT / "production",
    "soxl_runtime_core": ROOT / "src/soxl_quant",
    "soxl_policy": ROOT / "src/soxl_quant/policy",
    "soxl_app": ROOT / "src/soxl_quant/app",
    "soxl_tests": ROOT / "tests",
    "openalice_uta_service": OPENALICE / "services/uta",
    "openalice_protocol": OPENALICE / "packages/uta-protocol",
    "openalice_core": OPENALICE / "src/core",
    "openalice_workspaces": OPENALICE / "src/workspaces",
    "openalice_tools": OPENALICE / "src/tool",
    "openalice_task": OPENALICE / "src/task",
    "openalice_auth": OPENALICE / "src/services/auth",
}

rows = []
for name, path in interesting_roots.items():
    files = [p for p in path.rglob("*") if p.is_file() and p.suffix not in {".pyc", ".png", ".ico"}] if path.exists() else []
    source_files = [p for p in files if p.suffix in {".py", ".ts", ".tsx", ".js", ".mjs", ".sh", ".md", ".json", ".yaml", ".yml"}]
    line_count = 0
    for p in source_files:
        try:
            line_count += len(p.read_text(encoding="utf-8", errors="ignore").splitlines())
        except Exception:
            pass
    rows.append({"area": name, "files": len(files), "source_files": len(source_files), "source_lines": line_count})

inventory = pd.DataFrame(rows).sort_values("area")
display(inventory)
(CARD / "architecture_inventory.csv").write_text(inventory.to_csv(index=False), encoding="utf-8")


,area,files,source_files,source_lines
11,openalice_auth,5,5,623
7,openalice_core,47,47,11562
6,openalice_protocol,14,14,1531
10,openalice_task,11,11,1891
9,openalice_tools,9,9,1576
5,openalice_uta_service,117,117,24656
8,openalice_workspaces,46,45,5629
3,soxl_app,37,37,12487
2,soxl_policy,21,21,4861
0,soxl_production_entrypoints,22,18,4255


372

## OpenAlice Takeaways

The useful OpenAlice lessons are control-plane lessons, not a mandate to become a multi-asset AI agent platform.

Source references inspected:

- [README architecture](https://github.com/TraderAlice/OpenAlice/blob/master/README.md#architecture): two long-lived processes, Alice for decisions/research/surfaces, UTA for broker credentials/execution/snapshots.
- [UTA service main](https://github.com/TraderAlice/OpenAlice/blob/master/services/uta/src/main.ts): broker carrier owns trading dependencies, binds localhost, exposes health, trading routes, snapshot scheduler, FX, catalog refresh, and shutdown.
- [Trading routes](https://github.com/TraderAlice/OpenAlice/blob/master/services/uta/src/http/routes-trading.ts): typed HTTP boundary for account, positions, orders, staging, commit, reject, push, and snapshots.
- [Trading-as-Git protocol types](https://github.com/TraderAlice/OpenAlice/blob/master/packages/uta-protocol/src/types/git.ts): explicit staged operations, commits, operation results, wallet state, and sync result schemas.
- [Broker protocol types](https://github.com/TraderAlice/OpenAlice/blob/master/packages/uta-protocol/src/types/broker.ts): broker health, account info, positions, decimal-string monetary fields, and structured error classes.
- [Guardian supervisor](https://github.com/TraderAlice/OpenAlice/blob/master/scripts/guardian/prod.mjs): starts UTA first, gates Alice on UTA health, watches restart flag, cascades shutdown.
- [Docker packaging](https://github.com/TraderAlice/OpenAlice/blob/master/Dockerfile) and [compose](https://github.com/TraderAlice/OpenAlice/blob/master/docker-compose.yml): a self-hosted runtime image with persistent data volume and supervised long-lived processes.
- [CI workflow](https://github.com/TraderAlice/OpenAlice/blob/master/.github/workflows/ci.yml): install, build, test gate.
- [Auth token store](https://github.com/TraderAlice/OpenAlice/blob/master/src/services/auth/token-store.ts): local operator auth token is hashed on disk and plaintext is shown once.


In [3]:
gap_rows = [
    {
        "capability": "Broker credential isolation",
        "openalice_pattern": "Separate UTA service owns broker connections/credentials; Alice uses HTTP SDK only.",
        "soxl_current": "Futu/IBKR adapters run inside production runner process; credentials and decision/execution path share one process boundary.",
        "launch_risk": "A model/scheduler/runtime bug has direct broker session access; harder to restart broker layer independently.",
        "priority": "P0",
        "recommendation": "Introduce a SOXL execution-carrier service or daemon boundary before real-money live trading.",
    },
    {
        "capability": "Order lifecycle state machine",
        "openalice_pattern": "Stage -> commit -> manual approval push -> sync/reject, with commit history.",
        "soxl_current": "Risk gate calls place_target_position directly; summaries and CSVs exist, but no separate staged intent ledger/approval commit before broker push.",
        "launch_risk": "Harder to inspect/reject an intended order before it reaches live broker; audit is after-the-fact.",
        "priority": "P0",
        "recommendation": "Add repo-native order-intent ledger with staged/approved/submitted/filled/cancelled/reconciled states.",
    },
    {
        "capability": "Human-in-the-loop live approval",
        "openalice_pattern": "AI tradingPush cannot execute; it tells the user to approve through UI/connected channel.",
        "soxl_current": "Dry-run, risk gates, notifications, and broker flags exist; live/manual approval is not yet a first-class state transition.",
        "launch_risk": "Semi-auto can drift into auto-execution semantics before operator has a review surface.",
        "priority": "P0",
        "recommendation": "Make live broker execution default to approval-required; only explicit emergency/low-risk modes bypass, if any.",
    },
    {
        "capability": "Supervisor and service readiness",
        "openalice_pattern": "Guardian starts carrier first, polls health, restarts on config flag, cascades shutdown.",
        "soxl_current": "LaunchAgents, scheduler health, and preflights exist; no single supervisor owns broker daemon + scheduler + dashboard lifecycle.",
        "launch_risk": "Failure recovery depends on shell scripts and operator interpretation; broker reload is not a managed lifecycle.",
        "priority": "P0",
        "recommendation": "Create a production supervisor contract: start/check/restart/stop for broker carrier, scheduler, dashboard export, and health JSON.",
    },
    {
        "capability": "Account snapshots and reconciliation",
        "openalice_pattern": "Snapshot service takes periodic/event-driven snapshots and exposes equity curve.",
        "soxl_current": "AccountSnapshot/ReconciliationRecord exist and paper dashboard uses rows, but live broker snapshots are not a durable append-only ledger with recovery semantics.",
        "launch_risk": "External fills, partial fills, cancelled orders, and broker-side changes can create silent drift.",
        "priority": "P0",
        "recommendation": "Persist broker snapshots and reconciliation deltas as append-only runtime records independent of daily CSV dashboard rows.",
    },
    {
        "capability": "Structured broker errors and retry policy",
        "openalice_pattern": "BrokerError classifies CONFIG/AUTH/NETWORK/EXCHANGE/MARKET_CLOSED and maps transient vs permanent.",
        "soxl_current": "Provider transient snippets and health scripts exist; broker adapters return mostly strings and booleans.",
        "launch_risk": "Auth/config failures may be treated like transient runtime failures, or transient broker failures may block too broadly.",
        "priority": "P1",
        "recommendation": "Standardize BrokerHealth/BrokerError/OrderStatus enums and route all Futu/IBKR failures through them.",
    },
    {
        "capability": "Typed external protocol",
        "openalice_pattern": "uta-protocol package separates wire schemas/types from service and UI.",
        "soxl_current": "Dataclasses define runtime objects, but there is no versioned execution API schema for CLI/service/UI consumers.",
        "launch_risk": "Dashboard, runner, notifier, and future UI can disagree on field meanings.",
        "priority": "P1",
        "recommendation": "Publish a small versioned SOXL execution protocol under src/soxl_quant/core or runtime schema files.",
    },
    {
        "capability": "Operator UI / inbox surface",
        "openalice_pattern": "Workspace, inbox, Web UI, and portfolio dashboard are central user surfaces.",
        "soxl_current": "Static dashboard is strong for evidence; no operator order-review UI for staged intents.",
        "launch_risk": "Operator must inspect logs/JSON manually before execution, which is too brittle for live trading.",
        "priority": "P1",
        "recommendation": "Add a minimal local order-review dashboard/status page before building any broader app UI.",
    },
    {
        "capability": "Self-host packaging",
        "openalice_pattern": "Dockerfile + compose + persistent volume + CI build/test.",
        "soxl_current": "macOS LaunchAgents and Vercel static dashboard exist; no container/systemd package for full runtime.",
        "launch_risk": "Reproducible deployment depends on this laptop setup and ad-hoc env files.",
        "priority": "P1",
        "recommendation": "Package production stack as a single documented self-host target after carrier/order ledger exists.",
    },
    {
        "capability": "Security/auth surface",
        "openalice_pattern": "Admin token/session store with hashed token and tight file permissions.",
        "soxl_current": "Vercel dashboard/export auth state is tracked, but local operator/admin auth is not a product boundary.",
        "launch_risk": "If a local/remote operator UI is added, access control will lag behind the new surface.",
        "priority": "P2",
        "recommendation": "Only add auth after a real operator UI exists, but include it in launch checklist.",
    },
]

gap = pd.DataFrame(gap_rows)
display(gap)
gap.to_csv(CARD / "openalice_gap_matrix.csv", index=False)


,capability,openalice_pattern,soxl_current,launch_risk,priority,recommendation
0,Broker credential isolation,Separate UTA service owns broker connections/c...,Futu/IBKR adapters run inside production runne...,A model/scheduler/runtime bug has direct broke...,P0,Introduce a SOXL execution-carrier service or ...
1,Order lifecycle state machine,Stage -> commit -> manual approval push -> syn...,Risk gate calls place_target_position directly...,Harder to inspect/reject an intended order bef...,P0,Add repo-native order-intent ledger with stage...
2,Human-in-the-loop live approval,AI tradingPush cannot execute; it tells the us...,"Dry-run, risk gates, notifications, and broker...",Semi-auto can drift into auto-execution semant...,P0,Make live broker execution default to approval...
3,Supervisor and service readiness,"Guardian starts carrier first, polls health, r...","LaunchAgents, scheduler health, and preflights...",Failure recovery depends on shell scripts and ...,P0,Create a production supervisor contract: start...
4,Account snapshots and reconciliation,Snapshot service takes periodic/event-driven s...,AccountSnapshot/ReconciliationRecord exist and...,"External fills, partial fills, cancelled order...",P0,Persist broker snapshots and reconciliation de...
5,Structured broker errors and retry policy,BrokerError classifies CONFIG/AUTH/NETWORK/EXC...,Provider transient snippets and health scripts...,Auth/config failures may be treated like trans...,P1,Standardize BrokerHealth/BrokerError/OrderStat...
6,Typed external protocol,uta-protocol package separates wire schemas/ty...,"Dataclasses define runtime objects, but there ...","Dashboard, runner, notifier, and future UI can...",P1,Publish a small versioned SOXL execution proto...
7,Operator UI / inbox surface,"Workspace, inbox, Web UI, and portfolio dashbo...",Static dashboard is strong for evidence; no op...,Operator must inspect logs/JSON manually befor...,P1,Add a minimal local order-review dashboard/sta...
8,Self-host packaging,Dockerfile + compose + persistent volume + CI ...,macOS LaunchAgents and Vercel static dashboard...,Reproducible deployment depends on this laptop...,P1,Package production stack as a single documente...
9,Security/auth surface,Admin token/session store with hashed token an...,"Vercel dashboard/export auth state is tracked,...","If a local/remote operator UI is added, access...",P2,"Only add auth after a real operator UI exists,..."


In [4]:
phase_rows = [
    {
        "phase": "Phase 0 - Current state",
        "decision": "Do not rewrite the research harness.",
        "work": "Keep SOXL scope, qlib validation, runtime notebooks, promotion gates, scheduler health, and dashboard evidence as the foundation.",
        "exit_gate": "Existing check_harness plus production health continues to pass.",
    },
    {
        "phase": "Phase 1 - Live execution control plane",
        "decision": "Highest priority before real-money launch.",
        "work": "Order-intent ledger; approval-required state transition; broker error/status enums; append-only snapshots/reconciliation; emergency halt flag.",
        "exit_gate": "A generated target can be staged, inspected, rejected, approved, submitted, synced, and reconciled without direct model-to-broker execution.",
    },
    {
        "phase": "Phase 2 - Broker carrier boundary",
        "decision": "Adopt OpenAlice's UTA idea in SOXL-small form.",
        "work": "Small local SOXL execution carrier for Futu/IBKR, localhost-only API, health endpoint, restartable independently from scheduler/model runner.",
        "exit_gate": "Scheduler can decide and stage through a protocol; carrier owns broker sessions and can restart without changing model process.",
    },
    {
        "phase": "Phase 3 - Operator surface",
        "decision": "Build only what the operator needs, not a broad multi-asset app.",
        "work": "Local status/order-review page or static-plus-API dashboard; latest signal, staged order, approval/reject buttons, broker state, stale/health banners.",
        "exit_gate": "Operator can approve/reject live intent from one screen with full context and no log spelunking.",
    },
    {
        "phase": "Phase 4 - Packaging and runbook",
        "decision": "Make deployment reproducible after control plane is solid.",
        "work": "Self-host package, env validation, service supervisor, backup/restore, crash recovery, CI smoke, disaster manual-close runbook.",
        "exit_gate": "A clean machine can restore config/state, run health checks, and perform dry-run/full paper loop from documented steps.",
    },
    {
        "phase": "Phase 5 - Optional expansion",
        "decision": "Defer until SOXL launch path is stable.",
        "work": "Multi-account/multi-asset, workspace agents, inbox workflows, richer research tools.",
        "exit_gate": "Only start after SOXL live execution control plane has repeated evidence and operator confidence.",
    },
]

phases = pd.DataFrame(phase_rows)
display(phases)
phases.to_csv(CARD / "launch_readiness_phases.csv", index=False)


,phase,decision,work,exit_gate
0,Phase 0 - Current state,Do not rewrite the research harness.,"Keep SOXL scope, qlib validation, runtime note...",Existing check_harness plus production health ...
1,Phase 1 - Live execution control plane,Highest priority before real-money launch.,Order-intent ledger; approval-required state t...,"A generated target can be staged, inspected, r..."
2,Phase 2 - Broker carrier boundary,Adopt OpenAlice's UTA idea in SOXL-small form.,Small local SOXL execution carrier for Futu/IB...,Scheduler can decide and stage through a proto...
3,Phase 3 - Operator surface,"Build only what the operator needs, not a broa...",Local status/order-review page or static-plus-...,Operator can approve/reject live intent from o...
4,Phase 4 - Packaging and runbook,Make deployment reproducible after control pla...,"Self-host package, env validation, service sup...","A clean machine can restore config/state, run ..."
5,Phase 5 - Optional expansion,Defer until SOXL launch path is stable.,"Multi-account/multi-asset, workspace agents, i...",Only start after SOXL live execution control p...


## Direct Answer

现在不是“模型架构太简单”，而是**生产控制面还没有完整产品化**。SOXL repo 的研究侧、回测侧、promotion gate、runtime evidence、scheduler/dashboard 已经比很多个人量化项目更严格；距离上线主要差在 live trading 的安全边界、操作状态机、人工审批和可恢复运维。

最值得从 OpenAlice 学的 5 点：

1. **把 broker credential carrier 独立出来**：决策进程不直接持有 broker session。SOXL 可先做一个很小的本地 execution carrier，不需要完整 OpenAlice 平台。
2. **订单要先 stage/commit，再 push**：模型输出只是 intent；真正下单前要可审阅、可拒绝、可留 hash/ledger。
3. **manual approval 是 live 的默认边界**：尤其 SOXL 是 3x ETF，首版上线不该让 scheduler 直接自动打到 live broker。
4. **snapshot/reconcile 要独立于 dashboard CSV**：dashboard 是展示层，不应承担真实账户账本和恢复语义。
5. **supervisor/health/package 要变成产品边界**：现在 LaunchAgent/health 已经有基础，但还不是一个完整 restartable service stack。

## Keep / Do Not Copy

Keep:

- SOXL 单标的边界。
- qlib/research notebook-first workflow。
- promotion gate 和 runtime evidence discipline。
- 当前 scheduler health / Vercel dashboard 作为观察层。

Do not copy yet:

- OpenAlice 的多资产 UTA 复杂度。
- workspace/inbox/agent platform 大框架。
- 广泛市场数据和 AI provider 平台化。

## Single-Run Conclusion

Decision: **do not start another model/reward/network experiment for this concern**. The next meaningful architecture work should be a production/harness design for a SOXL live-execution control plane, starting with order intent ledger + approval-required state machine, then optionally a small broker carrier. This is a `P0` launch-readiness gap because it changes whether future live broker execution is reviewable and recoverable.

Primary sidecar artifacts written by this notebook:

- `architecture_inventory.csv`
- `openalice_gap_matrix.csv`
- `launch_readiness_phases.csv`
